In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import json, time
import numpy as np
import cv2
import matplotlib.pyplot as plt
from collections import Counter
from standard_e2e import Modality

TRAIN_DIR     = '../data/processed/waymo_e2e/training/'
MANIFEST_PATH = '../data/train_manifest.json'
FEATURES_DIR  = '../data/processed/waymo_e2e/features/'

# ALL 5 classes kept: the whole point is to test whether side-segment (adjacent-lane)
# features recover the lane-change signal that center/panorama features could not.
CLASSES = ['straight', 'left-turn', 'right-turn', 'lane-change-left', 'lane-change-right']
LANE_CHANGE = ['lane-change-left', 'lane-change-right']

# Segment boundaries in the 384-wide panorama. VERIFY left/right against seam detection.
SEGMENTS = {
    'left':   (0, 128),
    'center': (128, 256),
    'right':  (256, 384),
}

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)
print(f'{len(manifest)} sequences in manifest')

def load_img(fname):
    data = np.load(os.path.join(TRAIN_DIR, fname), allow_pickle=True)
    return np.array(data['_modality_data'].item()[Modality.CAMERAS])

In [ ]:
def detect_edges_and_lines(img, canny_low=30, canny_high=100, hough_threshold=15,
                            min_line_length=20, max_line_gap=15, road_region_frac=0.55):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    H, W = gray.shape
    road_mask = np.zeros_like(gray); road_mask[int(H * road_region_frac):, :] = 1
    edges = cv2.Canny(gray, canny_low, canny_high)
    edges_masked = edges * road_mask
    lines_raw = cv2.HoughLinesP(edges_masked, rho=1, theta=np.pi / 180, threshold=hough_threshold,
                                 minLineLength=min_line_length, maxLineGap=max_line_gap)
    lines = []
    if lines_raw is not None:
        for line in lines_raw:
            x1, y1, x2, y2 = line[0]
            angle = abs(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
            if 20 < angle < 75 or 105 < angle < 160:
                lines.append((x1, y1, x2, y2))
    return edges, np.array(lines)


def estimate_vanishing_point(lines, img_shape):
    H, W = img_shape[:2]
    def line_intersection(l1, l2):
        x1, y1, x2, y2 = l1; x3, y3, x4, y4 = l2
        denom = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
        if abs(denom) < 1e-6: return None
        t = ((x1 - x3) * (y3 - y4) - (y1 - y3) * (x3 - x4)) / denom
        return (x1 + t * (x2 - x1), y1 + t * (y2 - y1))
    if len(lines) < 2: return (0.5, 0.5), len(lines)
    pts = [p for i in range(len(lines)) for j in range(i + 1, len(lines))
           if (p := line_intersection(lines[i], lines[j])) and -W < p[0] < 2*W and -H < p[1] < 2*H]
    if not pts: return (0.5, 0.5), len(lines)
    xs, ys = np.array([p[0] for p in pts]), np.array([p[1] for p in pts])
    hist, xe, ye = np.histogram2d(xs, ys, bins=[np.linspace(-W, 2*W, 30), np.linspace(-H, 2*H, 30)])
    pi = np.unravel_index(hist.argmax(), hist.shape)
    return (((xe[pi[0]] + xe[pi[0]+1]) / 2) / W, ((ye[pi[1]] + ye[pi[1]+1]) / 2) / H), len(lines)


def get_robust_seed_color(hsv, H, W):
    pts = [(W//2, int(H*.95)), (W//2, int(H*.85)), (int(W*.35), int(H*.92)),
           (int(W*.65), int(H*.92)), (int(W*.35), int(H*.82)), (int(W*.65), int(H*.82))]
    colors, valid = [], []
    for sx, sy in pts:
        patch = hsv[max(0, sy-5):sy+5, max(0, sx-10):sx+10]
        if patch.size > 0:
            colors.append(np.mean(patch, axis=(0, 1))); valid.append((sx, sy))
    colors = np.array(colors)
    med = np.median(colors, axis=0)
    best = np.argmin(np.linalg.norm(colors - med, axis=1))
    return colors[best].astype(np.uint8), valid[best]


def segment_road(img, road_region_frac=0.55):
    H, W = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    seed_color, (sx, sy) = get_robust_seed_color(hsv, H, W)
    tol = np.array([20, 60, 60])
    mask = cv2.inRange(hsv, np.clip(seed_color.astype(int)-tol, 0, 255).astype(np.uint8),
                        np.clip(seed_color.astype(int)+tol, 0, 255).astype(np.uint8))
    mask[:int(H*road_region_frac), :] = 0
    flood = np.zeros((H+2, W+2), dtype=np.uint8)
    cv2.floodFill(mask.copy(), flood, (sx, sy), 255, loDiff=(10,10,10), upDiff=(10,10,10))
    road_mask = (flood[1:-1, 1:-1] * 255).astype(np.uint8)
    px = np.where(road_mask > 0)
    if len(px[0]) < 10:
        return road_mask, {'road_area_frac': 0., 'road_centroid_x': 0.5, 'road_taper': 0.}
    area = len(px[0]) / (H * W)
    cx = np.mean(px[1]) / W
    taper = (np.sum(road_mask[int(H*.9), :] > 0) - np.sum(road_mask[int(H*.7), :] > 0)) / W
    return road_mask, {'road_area_frac': area, 'road_centroid_x': cx, 'road_taper': taper}

In [ ]:
from ultralytics import YOLO
yolo_model = YOLO('yolov8s.pt')
VEHICLE_CLASSES = {2, 5, 7}  # car, bus, truck

def occupancy_in_crop(crop_img, model, is_center):
    """Vehicle occupancy within a single segment crop.
    Splits the crop into left/right thirds. On the CENTER segment the middle third is the
    ego lane and is excluded; on SIDE segments there is no ego lane, so all vehicles count.
    Returns: left_count, left_crowding, right_count, right_crowding, asymmetry.
    (crowding weights by how low the box sits = closer.)"""
    H, W = crop_img.shape[:2]
    res = model(crop_img, verbose=False)
    left, right = [], []
    for b in res[0].boxes:
        if int(b.cls) not in VEHICLE_CLASSES:
            continue
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        cx = ((x1 + x2) / 2) / W          # 0..1 within crop
        dx = cx - 0.5
        weight = y2 / H                    # bottom of box; larger = closer
        if is_center and abs(dx) <= 1/6:
            continue                       # ego lane (center segment only)
        if dx < 0:
            left.append(weight)
        else:
            right.append(weight)
    lc, rc = len(left), len(right)
    lcrowd = sum(left) if left else 0.0
    rcrowd = sum(right) if right else 0.0
    asym = (lcrowd - rcrowd)
    return [lc, lcrowd, rc, rcrowd, asym]

In [ ]:
# --- Per-segment feature vector ---
# For each segment: road geometry (centroid_x, area_frac, taper, vp_x, n_lines) + occupancy (5)
ROAD_KEYS = ['road_centroid_x', 'road_area_frac', 'road_taper', 'vp_x', 'n_lines']
OCC_KEYS  = ['occ_left_ct', 'occ_left_crowd', 'occ_right_ct', 'occ_right_crowd', 'occ_asym']
PER_SEG_KEYS = ROAD_KEYS + OCC_KEYS   # 10 features per segment

def segment_features(img, seg_bounds, is_center, model):
    """All features for one segment crop."""
    x0, x1 = seg_bounds
    crop = img[:, x0:x1]
    _, road = segment_road(crop)
    _, lines = detect_edges_and_lines(crop)
    (vp_x, _), n_lines = estimate_vanishing_point(lines, crop.shape)
    occ = occupancy_in_crop(crop, model, is_center)
    return [road['road_centroid_x'], road['road_area_frac'], road['road_taper'],
            vp_x, float(n_lines)] + occ

def all_segment_features(img, model):
    """Concatenate features across left/center/right. Returns dict segment->list."""
    out = {}
    for name, bounds in SEGMENTS.items():
        out[name] = segment_features(img, bounds, is_center=(name == 'center'), model=model)
    return out

In [ ]:
# --- Extract across all 3-class sequences (target frame only) ---
feats_by_seg = {'left': [], 'center': [], 'right': []}
labels, seq_ids = [], []

start = time.time()
n_total = sum(1 for e in manifest.values() if e['label'] in CLASSES)
done = 0
for sid, entry in manifest.items():
    if entry['label'] not in CLASSES:
        continue   # drop lane-changes (and stationary)
    img = load_img(entry['target_fname'])
    seg_feats = all_segment_features(img, yolo_model)
    for name in SEGMENTS:
        feats_by_seg[name].append(seg_feats[name])
    labels.append(entry['label'])
    seq_ids.append(sid)
    done += 1
    if done % 100 == 0:
        el = time.time() - start
        print(f'  {done}/{n_total} — {el/done*(n_total-done):.0f}s remaining...')

for name in SEGMENTS:
    feats_by_seg[name] = np.array(feats_by_seg[name], dtype=np.float32)
labels = np.array(labels)
seq_ids = np.array(seq_ids)
print(f'\nDone in {time.time()-start:.0f}s. Per-segment shape: {feats_by_seg["center"].shape}')
print('Class balance:', Counter(labels))

In [ ]:
# --- Save (own filenames, does not touch existing arrays) ---
for name in SEGMENTS:
    np.save(os.path.join(FEATURES_DIR, f'seg_{name}.npy'), feats_by_seg[name])
np.save(os.path.join(FEATURES_DIR, 'seg_labels.npy'), labels)
np.save(os.path.join(FEATURES_DIR, 'seg_seq_ids.npy'), seq_ids)
print('Saved seg_left/center/right.npy + labels + seq_ids')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

center = feats_by_seg['center']
sides  = np.concatenate([feats_by_seg['left'], feats_by_seg['right']], axis=1)
all_seg = np.concatenate([feats_by_seg['left'], feats_by_seg['center'], feats_by_seg['right']], axis=1)

def evaluate(X, y, tag):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
    sc = StandardScaler().fit(Xtr)
    print(f'\n=== {tag}  (dim={X.shape[1]}) ===')
    for nm, clf in [('SVM', SVC(kernel='rbf', random_state=0)),
                    ('RF', RandomForestClassifier(n_estimators=300, random_state=0))]:
        clf.fit(sc.transform(Xtr), ytr)
        pred = clf.predict(sc.transform(Xte))
        rep = classification_report(yte, pred, labels=CLASSES, output_dict=True, zero_division=0)
        print(f'  {nm}: acc={(pred==yte).mean():.3f}')
        for c in CLASSES:
            flag = '  <-- LANE CHANGE' if c in LANE_CHANGE else ''
            print(f'      {c:<20} recall={rep[c]["recall"]:.3f}{flag}')

# center-only reproduces the known failure; sides-only and all-segments test the fix
evaluate(center,  labels, 'center-only (baseline / known failure)')
evaluate(sides,   labels, 'sides-only (left+right, the adjacent-lane views)')
evaluate(all_seg, labels, 'all 3 segments')

In [ ]:
# --- Confusion matrix (all segments, RF) — do lane changes ever get predicted? ---
Xtr, Xte, ytr, yte = train_test_split(all_seg, labels, test_size=0.25, random_state=0, stratify=labels)
sc = StandardScaler().fit(Xtr)
clf = RandomForestClassifier(n_estimators=300, random_state=0).fit(sc.transform(Xtr), ytr)
pred = clf.predict(sc.transform(Xte))
cm = confusion_matrix(yte, pred, labels=CLASSES)
print('Confusion (all segments, RF)  [rows=true, cols=pred]:')
print(f'{"":>18}' + ''.join(f'{c[:8]:>10}' for c in CLASSES))
for i, c in enumerate(CLASSES):
    print(f'{c:>18}' + ''.join(f'{cm[i,j]:>10}' for j in range(len(CLASSES))))

# did ANY lane-change get predicted as a lane-change?
lc_idx = [CLASSES.index(c) for c in LANE_CHANGE]
lc_correct = sum(cm[i, i] for i in lc_idx)
lc_total = sum(cm[i, :].sum() for i in lc_idx)
print(f'\nLane-change: {lc_correct}/{lc_total} predicted correctly with side segments included.')

# feature importance by segment — are the SIDE segments contributing at all?
imp = clf.feature_importances_
print('\nFeature importance by segment (summed):')
for k, name in enumerate(SEGMENTS):
    seg_imp = imp[k*len(PER_SEG_KEYS):(k+1)*len(PER_SEG_KEYS)].sum()
    print(f'  {name:>7}: {seg_imp:.3f}')

# within side segments, is it the occupancy or the road-geometry features that matter?
print('\nSide-segment feature importance (left+right), road vs occupancy:')
for k, name in enumerate(SEGMENTS):
    if name == 'center':
        continue
    base = k * len(PER_SEG_KEYS)
    road_imp = imp[base:base+len(ROAD_KEYS)].sum()
    occ_imp  = imp[base+len(ROAD_KEYS):base+len(PER_SEG_KEYS)].sum()
    print(f'  {name:>7}: road={road_imp:.3f}  occupancy={occ_imp:.3f}')

In [ ]:
# --- Inspect the 3 correctly-predicted lane changes ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from standard_e2e import Modality, TrajectoryComponent
import matplotlib.pyplot as plt

# refit all-segments RF, track seq_ids into test split
idx = np.arange(len(all_seg))
Xtr, Xte, ytr, yte, itr, ite = train_test_split(
    all_seg, labels, idx, test_size=0.25, random_state=0, stratify=labels)
sc = StandardScaler().fit(Xtr)
clf = RandomForestClassifier(n_estimators=300, random_state=0).fit(sc.transform(Xtr), ytr)
pred = clf.predict(sc.transform(Xte))
test_sids = seq_ids[ite]

# find the correct lane-change predictions
correct_lc = [(test_sids[k], yte[k]) for k in range(len(yte))
              if yte[k] in LANE_CHANGE and pred[k] == yte[k]]
print(f'{len(correct_lc)} correctly-predicted lane changes:')
for sid, lbl in correct_lc:
    print(f'  {sid[:8]}  {lbl}')

# how often did the model predict each lane-change class at all? (base rate for chance)
from collections import Counter
pred_counts = Counter(pred)
print('\nModel prediction counts (test set):')
for c in CLASSES:
    print(f'  {c:<20} predicted {pred_counts.get(c,0)} times')

def load_img(fn):
    d = np.load(os.path.join(TRAIN_DIR, fn), allow_pickle=True)
    return np.array(d['_modality_data'].item()[Modality.CAMERAS])

# filmstrip + trajectory + lateral displacement for each
for sid, lbl in correct_lc:
    entry = manifest[sid]
    frames = entry['context_fnames']
    fidx = np.linspace(0, len(frames)-1, 5).astype(int)
    d = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
    fut = d['_modality_data'].item()[Modality.FUTURE_STATES]
    ys = fut.get(TrajectoryComponent.Y).flatten()
    lat = ys[-1] - ys[0]

    fig, axes = plt.subplots(1, 6, figsize=(17, 2.8))
    for j, fi in enumerate(fidx):
        axes[j].imshow(load_img(frames[fi])); axes[j].axis('off'); axes[j].set_title(f'f{fi}', fontsize=8)
    axes[5].plot(-ys, np.arange(len(ys)), '-o', ms=2)
    axes[5].set_title(f'lat={lat:+.1f}m', fontsize=9); axes[5].set_aspect('auto')
    fig.suptitle(f'{sid[:8]}  {lbl}  (correctly predicted)', fontsize=10, y=1.02)
    plt.tight_layout(); plt.show()

In [ ]:
# --- Review MISSED lane changes (for comparison against the 3 correct ones) ---
from standard_e2e import Modality, TrajectoryComponent
import matplotlib.pyplot as plt
import random

# reuse the fitted model / test split from the correct-prediction cell
# (all_seg, labels, seq_ids, and the train_test_split with itr/ite must already exist)
missed_lc = [(test_sids[k], yte[k], pred[k]) for k in range(len(yte))
             if yte[k] in LANE_CHANGE and pred[k] != yte[k]]
print(f'{len(missed_lc)} missed lane changes total')

# sample a handful, mixing left and right if possible
random.seed(1)
random.shuffle(missed_lc)
sample = missed_lc[:6]

def load_img(fn):
    d = np.load(os.path.join(TRAIN_DIR, fn), allow_pickle=True)
    return np.array(d['_modality_data'].item()[Modality.CAMERAS])

for sid, true, predicted in sample:
    entry = manifest[sid]
    frames = entry['context_fnames']
    fidx = np.linspace(0, len(frames)-1, 5).astype(int)
    d = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
    fut = d['_modality_data'].item()[Modality.FUTURE_STATES]
    ys = fut.get(TrajectoryComponent.Y).flatten()
    lat = ys[-1] - ys[0]

    fig, axes = plt.subplots(1, 6, figsize=(17, 2.8))
    for j, fi in enumerate(fidx):
        axes[j].imshow(load_img(frames[fi])); axes[j].axis('off'); axes[j].set_title(f'f{fi}', fontsize=8)
    axes[5].plot(-ys, np.arange(len(ys)), '-o', ms=2)
    axes[5].set_title(f'lat={lat:+.1f}m', fontsize=9)
    axes[5].axvline(0, color='gray', lw=0.5, ls='--')
    fig.suptitle(f'{sid[:8]}  TRUE {true} -> PRED {predicted}   (MISSED)', fontsize=10, y=1.02)
    plt.tight_layout(); plt.show()

# quick summary: what do the misses get predicted AS, and their displacement distribution
from collections import Counter
print('\nMissed lane changes predicted as:')
for c, n in Counter(p for _, _, p in missed_lc).most_common():
    print(f'  {c}: {n}')
lats = []
for sid, _, _ in missed_lc:
    d = np.load(os.path.join(TRAIN_DIR, manifest[sid]['target_fname']), allow_pickle=True)
    ys = d['_modality_data'].item()[Modality.FUTURE_STATES].get(TrajectoryComponent.Y).flatten()
    lats.append(abs(ys[-1]-ys[0]))
print(f'\nMissed lane-change |lat|: min={min(lats):.1f} max={max(lats):.1f} mean={np.mean(lats):.1f}m')

In [ ]:
# --- Build a GIF from ALL raw frames of a sequence on disk (not just manifest context) ---
import imageio.v2 as imageio
import numpy as np, os, glob, cv2, re

def load_img(fpath):
    d = np.load(fpath, allow_pickle=True)
    return np.array(d['_modality_data'].item()[Modality.CAMERAS])

def make_raw_sequence_gif(sid_prefix, out_dir='outputs', fps=4, upscale=2):
    # resolve full sequence hash
    full = [s for s in manifest if s.startswith(sid_prefix)][0]
    label = manifest[full]['label']

    # find ALL npz files for this sequence hash on disk
    pattern = os.path.join(TRAIN_DIR, f'{full}_*.npz')
    files = glob.glob(pattern)
    # sort by frame number (the integer after the last underscore)
    def frame_num(p):
        m = re.search(rf'{full}_(\d+)\.npz$', os.path.basename(p))
        return int(m.group(1)) if m else -1
    files = sorted([f for f in files if frame_num(f) >= 0], key=frame_num)
    print(f'{full[:12]}  {label}  found {len(files)} raw frames on disk '
          f'(manifest context was {manifest[full]["n_context"]})')

    imgs = []
    for f in files:
        img = load_img(f)
        if img.dtype != np.uint8:
            img = (img*255).astype(np.uint8) if img.max() <= 1 else np.clip(img,0,255).astype(np.uint8)
        if upscale > 1:
            img = cv2.resize(img, (img.shape[1]*upscale, img.shape[0]*upscale),
                             interpolation=cv2.INTER_NEAREST)
        imgs.append(img)

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f'rawseq_{sid_prefix}_{label}.gif')
    imageio.mimsave(out_path, imgs, fps=fps, loop=0)
    print(f'  -> {out_path}')
    return out_path

make_raw_sequence_gif('e9d68b1c')

In [ ]:


# --- Build a GIF from ALL raw frames of a sequence on disk (not just manifest context) ---
import imageio.v2 as imageio
import numpy as np, os, glob, cv2, re

def load_img(fpath):
    d = np.load(fpath, allow_pickle=True)
    return np.array(d['_modality_data'].item()[Modality.CAMERAS])

def make_raw_sequence_gif(sid_prefix, out_dir='outputs', fps=4, upscale=2):
    # resolve full sequence hash
    full = [s for s in manifest if s.startswith(sid_prefix)][0]
    label = manifest[full]['label']

    # find ALL npz files for this sequence hash on disk
    pattern = os.path.join(TRAIN_DIR, f'{full}_*.npz')
    files = glob.glob(pattern)
    # sort by frame number (the integer after the last underscore)
    def frame_num(p):
        m = re.search(rf'{full}_(\d+)\.npz$', os.path.basename(p))
        return int(m.group(1)) if m else -1
    files = sorted([f for f in files if frame_num(f) >= 0], key=frame_num)
    print(f'{full[:12]}  {label}  found {len(files)} raw frames on disk '
          f'(manifest context was {manifest[full]["n_context"]})')

    imgs = []
    for f in files:
        img = load_img(f)
        if img.dtype != np.uint8:
            img = (img*255).astype(np.uint8) if img.max() <= 1 else np.clip(img,0,255).astype(np.uint8)
        if upscale > 1:
            img = cv2.resize(img, (img.shape[1]*upscale, img.shape[0]*upscale),
                             interpolation=cv2.INTER_NEAREST)
        imgs.append(img)

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f'rawseq_{sid_prefix}_{label}.gif')
    imageio.mimsave(out_path, imgs, fps=fps, loop=0)
    print(f'  -> {out_path}')
    return out_path

make_raw_sequence_gif('c32624fb')

In [ ]:

# --- Build a GIF from ALL raw frames of a sequence on disk (not just manifest context) ---
import imageio.v2 as imageio
import numpy as np, os, glob, cv2, re

def load_img(fpath):
    d = np.load(fpath, allow_pickle=True)
    return np.array(d['_modality_data'].item()[Modality.CAMERAS])

def make_raw_sequence_gif(sid_prefix, out_dir='outputs', fps=4, upscale=2):
    # resolve full sequence hash
    full = [s for s in manifest if s.startswith(sid_prefix)][0]
    label = manifest[full]['label']

    # find ALL npz files for this sequence hash on disk
    pattern = os.path.join(TRAIN_DIR, f'{full}_*.npz')
    files = glob.glob(pattern)
    # sort by frame number (the integer after the last underscore)
    def frame_num(p):
        m = re.search(rf'{full}_(\d+)\.npz$', os.path.basename(p))
        return int(m.group(1)) if m else -1
    files = sorted([f for f in files if frame_num(f) >= 0], key=frame_num)
    print(f'{full[:12]}  {label}  found {len(files)} raw frames on disk '
          f'(manifest context was {manifest[full]["n_context"]})')

    imgs = []
    for f in files:
        img = load_img(f)
        if img.dtype != np.uint8:
            img = (img*255).astype(np.uint8) if img.max() <= 1 else np.clip(img,0,255).astype(np.uint8)
        if upscale > 1:
            img = cv2.resize(img, (img.shape[1]*upscale, img.shape[0]*upscale),
                             interpolation=cv2.INTER_NEAREST)
        imgs.append(img)

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f'rawseq_{sid_prefix}_{label}.gif')
    imageio.mimsave(out_path, imgs, fps=fps, loop=0)
    print(f'  -> {out_path}')
    return out_path

make_raw_sequence_gif('53983b15')





In [ ]:
from standard_e2e import Modality, TrajectoryComponent

def heading_change(xs, ys, min_disp=0.5):
    """Net heading change: angle between initial and final direction of travel,
    reconstructed from the X/Y path with displacement-gated tangents."""
    # initial tangent: first segment with enough displacement
    def tangent(idx_range):
        for i in idx_range:
            dx, dy = xs[i[1]]-xs[i[0]], ys[i[1]]-ys[i[0]]
            if np.hypot(dx, dy) >= min_disp:
                return np.arctan2(dy, dx)
        return None
    n = len(xs)
    start_t = tangent([(0, j) for j in range(1, n)])
    end_t   = tangent([(n-1-j, n-1) for j in range(1, n)])
    if start_t is None or end_t is None:
        return None
    d = np.degrees(end_t - start_t)
    return (d + 180) % 360 - 180   # wrap to [-180, 180]

# compute for all labeled sequences
import matplotlib.pyplot as plt
data = {c: [] for c in ['straight','left-turn','right-turn','lane-change-left','lane-change-right']}
for sid, e in manifest.items():
    if e['label'] not in data: continue
    d = np.load(os.path.join(TRAIN_DIR, e['target_fname']), allow_pickle=True)
    fut = d['_modality_data'].item()[Modality.FUTURE_STATES]
    xs = fut.get(TrajectoryComponent.X).flatten()
    ys = fut.get(TrajectoryComponent.Y).flatten()
    hc = heading_change(xs, ys)
    lat = ys[-1] - ys[0]
    if hc is not None:
        data[e['label']].append((abs(lat), abs(hc)))

fig, ax = plt.subplots(figsize=(9,6))
colors = {'straight':'gray','left-turn':'blue','right-turn':'green',
          'lane-change-left':'orange','lane-change-right':'red'}
for c, pts in data.items():
    if pts:
        pts = np.array(pts)
        ax.scatter(pts[:,0], pts[:,1], s=8, alpha=0.4, label=c, color=colors[c])
ax.axhline(30, color='k', ls='--', lw=0.5, label='30 deg (turn threshold)')
ax.axvline(2.5, color='k', ls=':', lw=0.5, label='2.5m')
ax.set_xlabel('|lateral displacement| (m)'); ax.set_ylabel('|heading change| (deg)')
ax.legend(fontsize=8); ax.set_title('Lateral displacement vs heading change by class')
plt.show()

# key question: what fraction of lane-changes have LARGE heading change (likely curve-following)?
for c in ['lane-change-left','lane-change-right']:
    pts = np.array(data[c])
    high_hc = (pts[:,1] > 30).sum()
    print(f'{c}: {high_hc}/{len(pts)} ({100*high_hc/len(pts):.0f}%) have heading change >30deg (likely curve-following, not true LC)')

In [ ]:
# --- 3-class headline: best existing features, lane-changes dropped ---
import numpy as np, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter

FEATURES_DIR = '../data/processed/waymo_e2e/features/'
CLASSES3 = ['straight', 'left-turn', 'right-turn']

# load committed arrays
hog   = np.load(os.path.join(FEATURES_DIR, 'hog.npy'))
yolo  = np.load(os.path.join(FEATURES_DIR, 'yolo.npy'))
road  = np.load(os.path.join(FEATURES_DIR, 'road.npy'))
labels_all = np.load(os.path.join(FEATURES_DIR, 'labels.npy'), allow_pickle=True)

# 3-class mask (drop lane-changes + stationary)
mask = np.isin(labels_all, CLASSES3)
y = labels_all[mask]
feats = {'HOG': hog[mask], 'YOLO': yolo[mask], 'Road': road[mask],
         'HOG+YOLO+Road': np.concatenate([hog[mask], yolo[mask], road[mask]], axis=1)}

# baseline to beat
counts = Counter(y)
majority = max(counts.values()) / len(y)
print(f'3-class dataset: {len(y)} sequences  {dict(counts)}')
print(f'Majority-class baseline: {majority:.3f}  (predict "{max(counts, key=counts.get)}")\n')

def evaluate(X, y, tag):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
    sc = StandardScaler().fit(Xtr)
    print(f'=== {tag}  (dim={X.shape[1]}) ===')
    for nm, clf in [('SVM', SVC(kernel='rbf', random_state=0)),
                    ('RF', RandomForestClassifier(n_estimators=300, random_state=0))]:
        clf.fit(sc.transform(Xtr), ytr)
        pred = clf.predict(sc.transform(Xte))
        rep = classification_report(yte, pred, labels=CLASSES3, output_dict=True, zero_division=0)
        rec = '  '.join(f'{c[:8]}={rep[c]["recall"]:.2f}' for c in CLASSES3)
        acc = (pred == yte).mean()
        beat = '✓' if acc > majority else '✗'
        print(f'  {nm}: acc={acc:.3f} {beat}baseline   {rec}')
    print()

for tag, X in feats.items():
    evaluate(X, y, tag)

In [ ]:
# --- Lane detection go/no-go: threshold + warp + sliding window on CLEAN daytime central crops ---
import cv2, numpy as np, matplotlib.pyplot as plt
from standard_e2e import Modality

TRAIN_DIR = '../data/processed/waymo_e2e/training/'
CENTER = (128, 256)

def load_img(fn):
    d = np.load(os.path.join(TRAIN_DIR, fn), allow_pickle=True)
    return np.array(d['_modality_data'].item()[Modality.CAMERAS])

def lane_pixel_mask(crop):
    """Step 1 (cheapest, most likely to work): threshold for white/yellow lane pixels.
    HLS lightness+saturation for yellow, high lightness for white, + Sobel-x gradient."""
    hls = cv2.cvtColor(crop, cv2.COLOR_RGB2HLS)
    L, S = hls[:,:,1], hls[:,:,2]
    # white: high lightness
    white = L > 180
    # yellow: mid-high lightness + saturation
    yellow = (L > 120) & (S > 80)
    # gradient on lightness (Sobel x)
    sx = cv2.Sobel(L, cv2.CV_64F, 1, 0, ksize=3)
    sx = np.abs(sx); sx = np.uint8(255*sx/ (sx.max()+1e-6))
    grad = sx > 40
    return (white | yellow | grad).astype(np.uint8)

# pick clean bright daytime straight frames
import random; random.seed(3)
cands = []
for sid, e in manifest.items():
    if e['label'] != 'straight': continue
    img = load_img(e['target_fname'])
    if img.mean() > 110:
        cands.append((sid, e, img.mean()))
    if len(cands) > 150: break
cands.sort(key=lambda c: -c[2])
picks = cands[:6]

# show: crop | thresholded lane pixels
fig, axes = plt.subplots(len(picks), 2, figsize=(8, 2.6*len(picks)))
for i, (sid, e, br) in enumerate(picks):
    crop = load_img(e['target_fname'])[:, CENTER[0]:CENTER[1]]
    mask = lane_pixel_mask(crop)
    axes[i,0].imshow(crop); axes[i,0].axis('off'); axes[i,0].set_title(f'{sid[:8]} crop', fontsize=8)
    axes[i,1].imshow(mask, cmap='gray'); axes[i,1].axis('off')
    axes[i,1].set_title(f'{mask.sum()} lane px', fontsize=8)
plt.suptitle('GO/NO-GO Step 1: do lane pixels threshold cleanly on clean daytime crops?', fontsize=11)
plt.tight_layout(); plt.savefig('outputs/lane_threshold_gonogo.png', dpi=110); plt.show()

In [ ]:
# --- Build a GIF from ALL raw frames of a sequence on disk (not just manifest context) ---
import imageio.v2 as imageio
import numpy as np, os, glob, cv2, re

def load_img(fpath):
    d = np.load(fpath, allow_pickle=True)
    return np.array(d['_modality_data'].item()[Modality.CAMERAS])

def make_raw_sequence_gif(sid_prefix, out_dir='outputs', fps=4, upscale=2):
    # resolve full sequence hash
    full = [s for s in manifest if s.startswith(sid_prefix)][0]
    label = manifest[full]['label']

    # find ALL npz files for this sequence hash on disk
    pattern = os.path.join(TRAIN_DIR, f'{full}_*.npz')
    files = glob.glob(pattern)
    # sort by frame number (the integer after the last underscore)
    def frame_num(p):
        m = re.search(rf'{full}_(\d+)\.npz$', os.path.basename(p))
        return int(m.group(1)) if m else -1
    files = sorted([f for f in files if frame_num(f) >= 0], key=frame_num)
    print(f'{full[:12]}  {label}  found {len(files)} raw frames on disk '
          f'(manifest context was {manifest[full]["n_context"]})')

    imgs = []
    for f in files:
        img = load_img(f)
        if img.dtype != np.uint8:
            img = (img*255).astype(np.uint8) if img.max() <= 1 else np.clip(img,0,255).astype(np.uint8)
        if upscale > 1:
            img = cv2.resize(img, (img.shape[1]*upscale, img.shape[0]*upscale),
                             interpolation=cv2.INTER_NEAREST)
        imgs.append(img)

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f'rawseq_{sid_prefix}_{label}.gif')
    imageio.mimsave(out_path, imgs, fps=fps, loop=0)
    print(f'  -> {out_path}')
    return out_path

make_raw_sequence_gif('dbd2d26c5b')